# Day 44 — Solutions

## Hierarchical clustering · DBSCAN

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | Building the tree | 1 – 3 |
| **B** | Linkage | 4 – 6 |
| **C** | DBSCAN | 7 – 10 |
| **D** | Choosing | 11 – 12 |

Run the setup cell first. Every question is self-contained after that.

> This is the solutions file. Each question shows working code and then the reasoning.
> Read the reasoning — most of the marks are there.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import load_iris, load_wine, make_moons, make_blobs, make_circles

TEAL, CORAL, VIOLET, BLUE, AMBER = "#1F9E86", "#E05C45", "#8268B5", "#3E7CB1", "#F0A530"
MUTED = "#7B8AA1"
COLS = [TEAL, CORAL, VIOLET, BLUE, AMBER, "#C96FA0", "#5BA87F"]

def show(X, labels=None, title="", ax=None):
    ax = ax or plt.gca()
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], c="#9AA8BE", s=18, edgecolors="white", linewidths=.4)
    else:
        for u in sorted(set(labels)):
            m = labels == u
            if u == -1:
                ax.scatter(X[m, 0], X[m, 1], c=MUTED, marker="x", s=22, linewidths=1.2)
            else:
                ax.scatter(X[m, 0], X[m, 1], c=COLS[u % 7], s=18,
                           edgecolors="white", linewidths=.4)
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([])

# ---- Part A works on six points you can check with a ruler ---------------
SIX = np.array([[0.0, 0.0], [0.5, 0.2], [4.0, 0.0], [4.4, 0.3], [2.0, 3.5], [2.3, 3.9]])

# ---- the rest uses the seeds-style wine data and some shapes -------------
WINE_X, WINE_Y = load_wine(return_X_y=True)
WINE_Z = StandardScaler().fit_transform(WINE_X)
print("six points:", SIX.shape, "   wine:", WINE_X.shape)

---
# Part A · Building the tree

## Question 1

Here are six points, laid out as three obvious pairs:

```
A (0.0, 0.0)   B (0.5, 0.2)      <- pair 1
C (4.0, 0.0)   D (4.4, 0.3)      <- pair 2
E (2.0, 3.5)   F (2.3, 3.9)      <- pair 3
```

Compute the full distance matrix yourself, print it, and say **which two points merge
first** and at what distance.

In [ ]:
D = np.sqrt(((SIX[:, None, :] - SIX[None, :, :]) ** 2).sum(axis=2))
print(pd.DataFrame(np.round(D, 3), index=list("ABCDEF"), columns=list("ABCDEF")))

off = D + np.eye(len(D)) * 999
i, j = np.unravel_index(off.argmin(), off.shape)
print("\nfirst merge:", "ABCDEF"[i], "and", "ABCDEF"[j], "at", round(off.min(), 4))

L6 = linkage(SIX, method="single")
print("scipy agrees, first merge at:", round(L6[0, 2], 4))

### Answer

The closest pair is **E and F**, at a distance of **0.5000**.

It is worth checking the other two pairs by eye: A–B is 0.5385 and C–D is 0.5000 as well
— so E–F and C–D are tied to four decimals, and the algorithm simply takes whichever it
meets first. Ties like this are normal and harmless, but they are the reason two
implementations can produce trees that differ in the order of early merges while giving
identical clusters.

Note that the three *pairs* merge before anything else happens, because within-pair
distances (~0.5) are far smaller than between-pair distances (~4). The tree will have
three tight twigs near the bottom and then three big joins near the top.

## Question 2

Build the full tree on the six points with `linkage(SIX, method='single')` and print the
merge matrix. Explain what each of the four columns means, and confirm the number of
merges is what you expect.

Then draw the dendrogram.

In [ ]:
L6 = linkage(SIX, method="single")
print(np.round(L6, 4))
print("\nmerges:", L6.shape[0], "  points:", len(SIX), "  n-1 =", len(SIX) - 1)

plt.figure(figsize=(6, 3.2))
dendrogram(L6, labels=list("ABCDEF"))
plt.ylabel("distance merged at"); plt.title("six points, single linkage")
plt.tight_layout(); plt.show()

### Answer

```
[[4.     5.     0.5    2.    ]    <- E+F        -> new cluster id 6
 [2.     3.     0.5    2.    ]    <- C+D        -> new cluster id 7
 [0.     1.     0.5385 2.    ]    <- A+B        -> new cluster id 8
 [7.     8.     3.5057 4.    ]    <- (CD)+(AB)  -> new cluster id 9
 [6.     9.     3.6249 6.    ]]   <- (EF)+that  -> everything
```

**The four columns are:** the first thing being merged, the second thing being merged,
the distance they were merged at, and how many original points the new cluster contains.

The ids are the key to reading it. Ids **0–5** are the original points A–F. Id **6** means
"the cluster created by merge row 0", id **7** means "the cluster created by merge row 1",
and so on. Row 4 says `6 and 9` — that is *the EF pair* joining *the four-point cluster
built in row 3*. A flat table of five rows has encoded a whole tree.

**Five merges for six points**, which is `n − 1` — always. Each merge reduces the number
of clusters by exactly one, and you start at 6 and end at 1. The last column confirms it:
2, 2, 2, 4, 6.

Look at the third column: **0.5, 0.5, 0.54, then 3.51, 3.62**. The three pairs form, and
then nothing happens for a long way up. That large empty band in the dendrogram is the
visual signature of three real groups — and it is the only thing you are reading when you
"cut the tree at the biggest gap".

## Question 3

Now the wine data: 178 wines, 13 scaled chemical measurements, 3 real cultivars.

Build a `ward` tree, then print for `k = 2, 3, 4, 5` the agreement with the true
cultivars and the silhouette. Also print the merge heights of the last five merges and
the jump between each.

Then explain: **what would you have concluded about `k` if nobody had told you there were
three cultivars?**

In [ ]:
LW = linkage(WINE_Z, method="ward")

print(f"{'k':>3}{'agreement':>12}{'silhouette':>13}")
for k in [2, 3, 4, 5]:
    lab = fcluster(LW, k, criterion="maxclust")
    print(f"{k:>3}{adjusted_rand_score(WINE_Y, lab):>12.4f}"
          f"{silhouette_score(WINE_Z, lab):>13.4f}")

heights = LW[:, 2]
print(f"\n{'clusters left':>14}{'merge height':>15}{'jump':>10}")
for i in range(1, 6):
    print(f"{i:>14}{heights[-i]:>15.2f}{heights[-i] - heights[-i-1]:>10.2f}")

plt.figure(figsize=(9, 3.2))
dendrogram(LW, no_labels=True, color_threshold=heights[-2])
plt.ylabel("distance merged at"); plt.title("178 wines, ward linkage")
plt.tight_layout(); plt.show()

### Answer

```
  k   agreement   silhouette
  2      0.4394       0.2670
  3      0.7899       0.2774
  4      0.6568       0.2258
  5      0.5830       0.1867

 clusters left   merge height      jump
             1          35.40       7.75
             2          27.65      15.08
             3          12.57       0.34
             4          12.23       0.51
             5          11.72       0.35
```

**You would have said three, and you would have been right — without ever looking at the
cultivar column.**

- **The silhouette peaks at k = 3** (0.2774). It is the only number in the first table
  that does not use the labels, and it picks the correct answer.
- **The jump column says the same thing, louder.** Read the row `2`: going from three
  clusters down to two costs **15.08**, while the merges below it cost 0.34, 0.51 and
  0.35 — forty times smaller. There is a cliff in the tree, and it sits exactly at three.
  In the dendrogram that is the tall empty band above the three coloured branches.
- Only then do we peek at the agreement column, which we are allowed to do because this
  is a teaching dataset: **0.7899 at k = 3**, far above every other k.

Two honest observations worth making:

1. **The silhouette values are all low** (0.19–0.28), far below the 0.74 we saw on the
   customer data. Thirteen dimensions is a roomy place and nothing is tightly packed. A
   low silhouette does not mean the clustering is wrong — it means you must compare
   silhouettes *across k on one dataset*, never across different datasets.
2. **The two methods are not independent, but they are not the same either.** The
   silhouette's margin is thin (0.2774 against 0.2670 — you would not bet much on that
   alone). The dendrogram's gap is enormous. When a weak signal and a strong signal point
   the same way, you have a real answer; when they disagree, you have a question for a
   domain expert, not a tie-break rule.

---
# Part B · Linkage

## Question 4

Run all four linkages on **three** different datasets at the right `k`:

- `make_blobs(300, centers=3, cluster_std=1.0, random_state=42)` — k = 3
- `make_moons(300, noise=0.06, random_state=0)` — k = 2
- `make_circles(300, factor=0.45, noise=0.05, random_state=0)` — k = 2

Print a table of agreements: four linkages down, three datasets across.

In [ ]:
SETS = [("blobs", *make_blobs(300, centers=3, cluster_std=1.0, random_state=42), 3),
        ("moons", *make_moons(300, noise=0.06, random_state=0), 2),
        ("circles", *make_circles(300, factor=0.45, noise=0.05, random_state=0), 2)]

out = {}
for lk in ["ward", "complete", "average", "single"]:
    out[lk] = {}
    for nm, D, y, k in SETS:
        Zs = StandardScaler().fit_transform(D)
        lab = AgglomerativeClustering(k, linkage=lk).fit(Zs).labels_
        out[lk][nm] = round(adjusted_rand_score(y, lab), 4)

print(pd.DataFrame(out).T)

fig, axes = plt.subplots(4, 3, figsize=(10, 11))
for r, lk in enumerate(["ward", "complete", "average", "single"]):
    for c, (nm, D, y, k) in enumerate(SETS):
        Zs = StandardScaler().fit_transform(D)
        lab = AgglomerativeClustering(k, linkage=lk).fit(Zs).labels_
        show(Zs, lab, f"{lk} / {nm}: {adjusted_rand_score(y, lab):.3f}", ax=axes[r][c])
plt.tight_layout(); plt.show()

### Answer

```
           blobs   moons  circles
ward         1.0  0.2090  -0.0002
complete     1.0  0.6176   0.0294
average      1.0  0.5462   0.0732
single       1.0  1.0000   1.0000
```

**On the blobs all four score exactly 1.0000.** When the groups are genuinely separate
round clumps, your choice of linkage does not matter at all — and this is the case most
textbook examples show you, which is why linkage gets treated as a footnote.

**On the moons and circles the column collapses.** Single linkage is perfect on both.
Ward, which was flawless on the blobs, scores **0.2090** on the moons and **−0.0002** on
the circles — and a negative agreement means it did *worse than labelling at random*.

The circles column is the clearest demonstration. Two concentric rings have the **same
centre**. Any method that reasons about centres or averages has literally nothing to work
with, so ward and complete split the picture down the middle like a pie and get both
rings half right, which scores zero. Single linkage never asks where the centre is — it
only ever asks "is the next point close?" — so it walks around each ring and stops at the
gap.

**Same four algorithms, same `k`, agreements from 1.0000 to −0.0002.** Linkage is not a
footnote.

## Question 5

Now find the case where single linkage is the **worst** choice.

Build two groups of different density — 120 tight points and 120 loose ones — and run all
four linkages at `k = 2`.

In [ ]:
r = np.random.default_rng(1)
XU = np.vstack([r.normal([0, 0], 0.28, (120, 2)),      # tight
                r.normal([3.8, 0], 1.10, (120, 2))])   # loose
YU = np.r_[np.zeros(120), np.ones(120)].astype(int)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, lk in zip(axes, ["ward", "complete", "average", "single"]):
    lab = AgglomerativeClustering(2, linkage=lk).fit(XU).labels_
    print(f"  {lk:9} agreement {adjusted_rand_score(YU, lab):.4f}"
          f"   sizes {sorted(np.bincount(lab).tolist())}")
    show(XU, lab, f"{lk}  {adjusted_rand_score(YU, lab):.3f}", ax=ax)
plt.tight_layout(); plt.show()

print("\ntight cluster reaches out to x =", round(XU[:120, 0].max(), 3))
print("loose cluster starts at      x =", round(XU[120:, 0].min(), 3))

### Answer

```
  ward      agreement 1.0000   sizes [120, 120]
  complete  agreement 0.3478   sizes [71, 169]
  average   agreement 0.9833   sizes [119, 121]
  single    agreement 0.0000   sizes [1, 239]
```

**Single linkage puts 239 of the 240 points in one cluster and leaves one point alone.**
Agreement **0.0000** — on the dataset where ward scores a perfect 1.0000.

The `sizes` column is where the diagnosis lives, and the last two printed lines explain
it: the tight cluster reaches out to **x = 0.713** and the loose cluster starts at
**x = 1.097**. They nearly touch. Single linkage merges two groups on the distance between
their **two closest members**, so that one narrow 0.38-wide gap is all it needs — and the
gaps *inside* the loose cluster are wider than that. It hops across and swallows
everything, then has to produce a second cluster from whatever is left, which is one
isolated outlier.

This is **chaining**, and it is single linkage's defining failure.

**Watch complete linkage too, at 0.3478 with sizes [71, 169].** It did not chain; it
committed the opposite error. Complete linkage measures groups by their two *furthest*
members, so it hates clusters with a big diameter — and it sliced the sprawling loose
cluster in half rather than tolerate one wide group. Average linkage, which sits between
the two extremes, got 0.9833.

**So single linkage is the best linkage on curved shapes (Q4) and the worst on this one.**
It is not a safe default; it is a specialist tool.

## Question 6

You have now seen single linkage score **1.000** on three datasets and **0.000** on a
fourth.

Explain in your own words what single linkage actually measures, why that makes it
brilliant on the moons and circles, and why the same property destroys it on the uneven
density data. Then say which linkage you would use by default, and why.

### Answer

**What it measures.** Single linkage defines the distance between two groups as the
distance between their **two closest members** — the single shortest hop between them. It
does not care how big the groups are, how spread out they are, or how far apart their
centres sit. It asks one question: *is there anywhere these two groups almost touch?*

**Why that is brilliant on the moons and circles.** A crescent is a long, thin, connected
shape. Its two ends are far apart, so any method that compares centres or averages (ward,
complete, average) sees one crescent as two distant things and happily splits it, while
grouping the facing tips of *different* crescents together. Single linkage instead walks
along the crescent in short hops — each point is close to its neighbour, which is close to
its neighbour — and the whole shape merges as one. The gap *between* the crescents is
wider than any hop *along* them, so it never crosses. Perfect result, and for the right
reason.

**Why the same property destroys it on uneven density.** The property is "I will merge two
groups if any single pair of points is close". That is only safe when the gap *between*
real groups is wider than the gaps *inside* them — and in Q5 it was not. The loose cluster
sprawls, so its own internal gaps are large, while its nearest edge sits 0.3 away from the
tight cluster. One short hop is a bridge, and single linkage crosses it and fuses
everything. **A single pair of points can do this**; the method has no notion of weight of
evidence, and more data does not fix it, because more data means more chances to find a
bridge.

Those are the same sentence. *"Merge if anything almost touches"* is a brilliant rule when
the clusters are connected shapes separated by clean gaps, and a disastrous one when they
are blobs of different spread. The method did not behave differently in the two cases —
the data did.

**What I would use by default: `ward`.** It scored 1.000, 0.209, −0.000 and 1.000 across
the four datasets — never the best on the curved shapes, but only ever badly wrong on the
circles, where every centre-based method is. It is the one of the four that behaves
sensibly when clusters differ in size and density, which is the common real case, and it
tends to produce clusters of comparable size, which is usually what people expect from a
segmentation.

Reach for `single` deliberately, when you have *looked at the data* and can see that the
groups are long connected shapes with clean gaps between them. Never reach for it blind.

---
# Part C · DBSCAN

## Question 7

Work out the three kinds of point **yourself**, without using `DBSCAN`.

For the dataset below with `eps = 0.5` and `min_samples = 5`:

- a point is **core** if at least `min_samples` points (including itself) lie within `eps`
- a point is **border** if it is not core but lies within `eps` of some core point
- otherwise it is **noise**

Print the three counts, then check them against sklearn.

In [ ]:
rg = np.random.default_rng(9)
PT = np.vstack([rg.normal([0, 0], 0.5, (60, 2)),
                rg.normal([3.0, 2.0], 0.5, (55, 2)),
                rg.uniform([-2, -2], [5, 4], (15, 2))])
EPS, MS = 0.5, 5

D = np.sqrt(((PT[:, None, :] - PT[None, :, :]) ** 2).sum(axis=2))
within = D <= EPS
is_core = within.sum(axis=1) >= MS
is_border = (~is_core) & within[:, is_core].any(axis=1)
is_noise = (~is_core) & (~is_border)

print("core  ", is_core.sum())
print("border", is_border.sum())
print("noise ", is_noise.sum())
print("total ", is_core.sum() + is_border.sum() + is_noise.sum(), "of", len(PT))

db = DBSCAN(eps=EPS, min_samples=MS).fit(PT)
sk_core = np.zeros(len(PT), bool); sk_core[db.core_sample_indices_] = True
print("\nmy core matches sklearn :", bool((sk_core == is_core).all()))
print("my noise matches sklearn:", bool(((db.labels_ == -1) == is_noise).all()))
print("clusters found          :", len(set(db.labels_)) - (1 if -1 in db.labels_ else 0))

plt.figure(figsize=(6, 4.4))
plt.scatter(*PT[is_core].T, c=TEAL, s=30, label="core")
plt.scatter(*PT[is_border].T, c=AMBER, s=30, label="border")
plt.scatter(*PT[is_noise].T, c=CORAL, marker="x", s=44, linewidths=2, label="noise")
plt.legend(fontsize=9, frameon=False); plt.xticks([]); plt.yticks([])
plt.title("core, border and noise, computed by hand"); plt.show()

### Answer

**core 111, border 5, noise 14** — they add to 130, and both core and noise match sklearn
exactly. You have just reimplemented the part of DBSCAN that matters, in four lines.

The `within[:, is_core].any(axis=1)` line is the one worth understanding. `within` is a
130×130 true/false table of "are these two points within eps". Slicing its columns to the
core points and asking `.any(axis=1)` asks, for every point, *"is there any core point
within eps of me?"* — which is exactly the definition of a border point, once you already
know the point is not core itself.

Two things to notice:

- **Border points are rare (5 of 130).** A real cluster is mostly interior. Border points
  live on the rim: close enough to belong, not crowded enough to extend the cluster
  further. This is why DBSCAN's clusters stop where they do.
- **Two clusters were found, and nobody passed a `k`.** No K-Means run could have told you
  anything was a stray — it would have assigned every point to a cluster and shifted a
  centre to accommodate it.

**But check the 14 noise points against what we actually planted, because the tidy story
is wrong:**

```
of the 15 scattered strays, called noise : 11     (4 were absorbed into a blob)
of the 115 blob members, called noise    :  3
                                           ---
                                            14
```

**DBSCAN did not simply find our strays.** It missed 4 of them and wrongly condemned 3
real blob members. That is not a bug — it is the definition doing its job. A "stray" that
lands by chance in the middle of a dense blob *is* in a dense neighbourhood, and DBSCAN is
right to absorb it; a blob member that lands out on the thin edge *is* in a sparse
neighbourhood. **DBSCAN labels position, not provenance.** Keep that in mind the next time
you are tempted to read `-1` as "this row is bad data".

## Question 8

Break it with `eps`. On `make_moons(300, noise=0.06, random_state=0)` sweep `eps` over
`[0.05, 0.10, 0.15, 0.20, 0.30, 0.50]` with `min_samples=5`, and print the number of
clusters found, the number of noise points, and the agreement.

In [ ]:
XM, YM = make_moons(300, noise=0.06, random_state=0)

print(f"{'eps':>6}{'clusters':>10}{'noise':>8}{'agreement':>12}")
for eps in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    d = DBSCAN(eps=eps, min_samples=5).fit(XM)
    nc = len(set(d.labels_)) - (1 if -1 in d.labels_ else 0)
    print(f"{eps:>6}{nc:>10}{(d.labels_ == -1).sum():>8}"
          f"{adjusted_rand_score(YM, d.labels_):>12.4f}")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, eps in zip(axes, [0.05, 0.10, 0.20, 0.30]):
    d = DBSCAN(eps=eps, min_samples=5).fit(XM)
    show(XM, d.labels_, f"eps={eps}  {adjusted_rand_score(YM, d.labels_):.3f}", ax=ax)
plt.tight_layout(); plt.show()

### Answer

```
   eps  clusters   noise   agreement
  0.05        10     243      0.0091
  0.10        11      15      0.3369
  0.15         2       2      0.9867
  0.20         2       0      1.0000
  0.30         1       0      0.0000
  0.50         1       0      0.0000
```

**There is no gentle failure.** Read the two ends:

- **Too small (0.05).** Almost nothing has five neighbours within such a tiny radius, so
  243 of the 300 points are declared noise and the survivors form ten fragments.
- **Too large (0.30).** The gap between the two crescents gets bridged, everything becomes
  one cluster, and agreement falls to exactly **0.000** — which is what you score for
  putting everything in one group.

The window that works (0.15 – 0.20) is narrow, and on either side of it the answer is not
*slightly* worse but completely wrong. That is the real cost of DBSCAN: you have traded
"choose k", where every value gives you a usable answer, for "choose eps", where most
values give you rubbish.

## Question 9

Use the **k-distance recipe** to choose `eps` without looking at the labels: measure each
point's distance to its 5th nearest neighbour, sort them, take the 90th percentile.

Apply it to the moons. Compare the agreement you get to the best value from Question 8,
and explain what the recipe is and is not good for.

In [ ]:
dist, _ = NearestNeighbors(n_neighbors=5).fit(XM).kneighbors(XM)
kd = np.sort(dist[:, -1])
suggested = float(np.quantile(kd, 0.90))

plt.figure(figsize=(6, 3))
plt.plot(kd, color=BLUE, lw=2)
plt.axhline(suggested, color=CORAL, ls="--")
plt.text(2, suggested + .006, f"90th percentile = {suggested:.3f}",
         color=CORAL, fontsize=9, fontweight="bold")
plt.xlabel("points, sorted"); plt.ylabel("distance to 5th neighbour")
plt.title("the k-distance plot"); plt.tight_layout(); plt.show()

a_sug = adjusted_rand_score(YM, DBSCAN(eps=suggested, min_samples=5).fit(XM).labels_)
a_best = adjusted_rand_score(YM, DBSCAN(eps=0.20, min_samples=5).fit(XM).labels_)
print("recipe suggests eps =", round(suggested, 4))
print("agreement there     =", round(a_sug, 4))
print("best from Q8 (0.20) =", round(a_best, 4))

### Answer

The recipe suggests **eps = 0.1135**, which scores **0.7050**. The value that actually
works is **0.20**, scoring **1.0000**.

**What the recipe is.** Most points in a real cluster sit in a crowd, so their distance to
their 5th neighbour is small and similar — that is the long flat stretch of the curve.
Points on the fringe have to reach much further, which is the sharp upturn at the right.
The elbow between the two is roughly the radius that separates "in a crowd" from "out on
your own", which is exactly what `eps` is supposed to mean.

**What it is good for: getting the order of magnitude right.** It told us 0.11 when the
answer was 0.20 — the same ballpark. Without it you would have no idea whether to try
0.01, 0.1 or 10, and on data with more than two columns you cannot simply look.

**What it is not good for: giving you the answer.** 0.7050 against 1.0000 is the
difference between a clustering you would ship and one you would not. The 90th percentile
is an arbitrary choice dressed up as a principle — the elbow on a real curve is rarely
sharp, and reading it off is a judgement call.

**So treat the recipe as a starting point and then sweep a handful of values around it,
plotting each one.** That is not a weakness of this particular heuristic; it is the
honest state of choosing `eps`, and it is the main practical reason DBSCAN is harder to
use than K-Means despite being the better idea.

## Question 10

Find a dataset where DBSCAN **loses to K-Means**.

Build two groups of different density again — 120 tight points and 120 much looser ones.
Try `eps` of 0.25, 0.45 and 0.80, and compare the best of them with `KMeans(2)`. Print the
noise count for each.

In [ ]:
rd = np.random.default_rng(5)
XD = np.vstack([rd.normal([0, 0], 0.30, (120, 2)),
                rd.normal([4.2, 0], 1.25, (120, 2))])
YD = np.r_[np.zeros(120), np.ones(120)].astype(int)

print(f"{'eps':>8}{'clusters':>10}{'noise':>8}{'agreement':>12}")
for eps in [0.25, 0.45, 0.80]:
    d = DBSCAN(eps=eps, min_samples=5).fit(XD)
    nc = len(set(d.labels_)) - (1 if -1 in d.labels_ else 0)
    print(f"{eps:>8}{nc:>10}{(d.labels_ == -1).sum():>8}"
          f"{adjusted_rand_score(YD, d.labels_):>12.4f}")

km = KMeans(2, n_init=10, random_state=0).fit(XD)
print(f"{'K-Means':>8}{2:>10}{'--':>8}{adjusted_rand_score(YD, km.labels_):>12.4f}")

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, eps in zip(axes[:3], [0.25, 0.45, 0.80]):
    d = DBSCAN(eps=eps, min_samples=5).fit(XD)
    show(XD, d.labels_, f"DBSCAN eps={eps}  {adjusted_rand_score(YD, d.labels_):.3f}",
         ax=ax)
show(XD, km.labels_, f"K-Means  {adjusted_rand_score(YD, km.labels_):.3f}", ax=axes[3])
plt.tight_layout(); plt.show()

### Answer

```
     eps  clusters   noise   agreement
    0.25         2     117      0.9274
    0.45         2      39      0.7655
    0.80         2       8      0.9214
 K-Means         2      --      0.9342
```

**K-Means wins, at 0.9342 against DBSCAN's best of 0.9274.**

The `noise` column explains why. DBSCAN applies **one radius to the whole dataset**. The
tight group has a spread of 0.30 and the loose one 1.25 — four times looser. To use a
radius small enough that the tight group stays intact, `eps = 0.25` has to declare
**117 of the 240 points** noise: essentially the entire loose cluster. Open the radius up
to 0.80 and the loose group survives, but now the tight group's neighbourhood reaches well
beyond it.

Note also that the middle value is the *worst* (0.7655), so this is not even a clean
trade-off you could tune your way along.

**The honest conclusion: the newer, cleverer method is not automatically the better one.**
DBSCAN beats K-Means on shape and loses to it on varying density. Neither fact is a
general ranking — they are facts about particular datasets, which is why you plot the data
before choosing.

---
# Part D · Choosing

## Question 11

Put everything together. For each of these three datasets, run **K-Means, ward, single
linkage and DBSCAN** and print a table of agreements.

- the wine data (`WINE_Z`, k = 3)
- `make_circles(300, factor=0.45, noise=0.05, random_state=0)` (k = 2)
- the uneven-density data `XD` (k = 2)

Choose DBSCAN's `eps` with the k-distance recipe each time, so no method gets to peek at
the answers.

In [ ]:
def eps_from_kdistance(D, min_samples=5, q=0.90):
    d, _ = NearestNeighbors(n_neighbors=min_samples).fit(D).kneighbors(D)
    return float(np.quantile(np.sort(d[:, -1]), q))

JOBS = [("wine", WINE_Z, WINE_Y, 3),
        ("circles", *make_circles(300, factor=0.45, noise=0.05, random_state=0), 2),
        ("uneven density", XD, YD, 2)]

out = {}
for nm, D, y, k in JOBS:
    Zs = StandardScaler().fit_transform(D)
    dl = DBSCAN(eps=eps_from_kdistance(Zs), min_samples=5).fit(Zs).labels_
    out[nm] = {
        "kmeans": round(adjusted_rand_score(
            y, KMeans(k, n_init=10, random_state=0).fit(Zs).labels_), 4),
        "ward": round(adjusted_rand_score(
            y, AgglomerativeClustering(k, linkage="ward").fit(Zs).labels_), 4),
        "single": round(adjusted_rand_score(
            y, AgglomerativeClustering(k, linkage="single").fit(Zs).labels_), 4),
        "dbscan": round(adjusted_rand_score(y, dl), 4),
    }
    out[nm]["dbscan_k_found"] = len(set(dl)) - (1 if -1 in dl else 0)
    out[nm]["dbscan_noise"] = int((dl == -1).sum())

print(pd.DataFrame(out))

### Answer

```
                  wine  circles  uneven density
kmeans          0.8975  -0.0030          0.9342
ward            0.7899  -0.0002          0.9833
single         -0.0068   1.0000          0.0001
dbscan         -0.0074   0.5790          0.9005
dbscan_k_found  1.0000  10.0000          2.0000
dbscan_noise    9.0000   4.0000         11.0000
```

**Every method is best on exactly one dataset and terrible on at least one other.**

- **wine** → K-Means (0.8975), with ward close behind. DBSCAN scored **−0.0074** and the
  `dbscan_k_found` row says why: it found **one** cluster and called 9 points noise. In 13
  dimensions everything is far from everything, so the k-distance radius that covers 90%
  of points covers essentially all of them at once. Single linkage, at −0.0068, chained
  for the same reason.
- **circles** → single linkage, **1.0000** outright, while K-Means and ward sit at about
  zero. DBSCAN managed 0.5790 — the k-distance eps was too small, so it shattered the
  rings into **10** fragments.
- **uneven density** → ward (0.9833), with single linkage at 0.0001.

**The practical lesson is in the two extra rows.** Agreement alone would tell you DBSCAN
failed on wine; `dbscan_k_found = 1` tells you *how*, and that is the thing you could have
noticed without any labels at all. Always print how many clusters DBSCAN actually found
and how many points it threw away — those two numbers are available on real data, where
the agreement column does not exist.

## Question 12

A colleague gives you 8,000 rows of customer data with 6 numeric columns. They tell you:

> *"I ran K-Means with k=4 because that's what we always use. Here are the four segments."*

Write **three or four sentences** recommending what they should do instead, naming the
specific checks and which methods you would try. Then say which single piece of evidence
would most change your recommendation.

### Answer

**A recommendation that works:**

> *"Before trusting those four segments, scale the six columns and check that `k=4` is
> actually supported — run the silhouette across k=2 to 8, and fit a ward tree so you can
> see every k from one fit and look for a big vertical gap in the dendrogram. 8,000 rows
> is comfortably within hierarchical clustering's reach, so that costs you one fit. I'd
> also re-run K-Means from several seeds to check the four segments are stable rather than
> an artefact of one random start, and try DBSCAN to see whether some customers are
> genuinely outliers that K-Means has been forcing into a segment. Then put the cluster
> profiles in front of someone from marketing: if they can't name the groups, the
> clustering hasn't found customers, it's found geometry."*

The specific checks, and why each one:

| Check | Why |
|---|---|
| **Scale first** | All four methods measure distance. Six columns in different units means the biggest-numbered one decides everything (Day 43: 0.467 → 0.975). |
| **Silhouette across k** | The only label-free evidence about `k`. "What we always use" is not evidence. |
| **A ward dendrogram** | One fit shows every k at once, and a big vertical gap suggests k directly. At 8,000 rows this is affordable; at 100,000 it would not be. |
| **Re-run with different seeds** | K-Means gets stuck. If the four segments move between runs, they are not real. |
| **Try DBSCAN** | It is the only method that can say "this customer belongs to no segment", and forced-in outliers distort K-Means centres. |
| **Show a human** | No metric can tell you the groups are the *right* ones. |

**What would most change the recommendation: a plot of the data.** Reduce the six columns
to two with PCA and look at it. If you see four round clouds, your colleague was right and
all of the above just confirms it in twenty minutes. If you see two clouds, or curved
bands, or one big cloud with a few stragglers, then `k=4` was never going to work and
which method to use follows immediately from the shape you can see.

A second, cheaper thing that would change it: **what the segments are for**. If they drive
a four-way email campaign that already exists, then four groups is a business constraint
and the honest question is "are these the best four?", not "is four right". Clustering
serves a decision; the decision sometimes fixes `k`.

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload the
notebook. Questions 3, 6, 9 and 12 are marked mostly on the explanation — a correct number
with no reasoning gets little credit.